# 13 — Unificación de bases (WDI + V-Dem + Market Concentration)

**Objetivo general del cuaderno:** construir una base única, en formato largo, que contenga
únicamente los indicadores presentes en `LLM_dataset_final.xlsx` (clasificación final vía LLM +
revisión manual), acotada a:

- Años 2000–2020
- Los 193 países miembros de la ONU (mismo criterio de estandarización usado en los cuadernos 02, 06 y 09)

Cada fila del resultado final representa una observación país–año–indicador, con su `fuente`
(WDI / V-Dem / MC) y su `dimension_principal` (MONETARIA / INSTITUCIONAL / ESTRUCTURAL /
SOCIODEMOGRAFICA) anexadas.

El resultado en formato largo puede alcanzar varios millones de filas (el caso
extremo es WDI: hasta 193 × 21 × 975 ≈ 3,9M filas). Esto excede el límite de una hoja de Excel
(1.048.576 filas), por lo que el archivo final se exporta en **CSV**, no en `.xlsx`.




## 1. Configuración e importaciones

In [1]:
import pandas as pd
import numpy as np
import zipfile
import requests
from pathlib import Path

pd.set_option("display.max_columns", None)


## 2. Metadata de indicadores (filtro maestro)

**Objetivo:** cargar `LLM_dataset_final.xlsx` y construir la tabla de metadata (`codigo`,
`fuente`, `dimension_principal`) que se anexará a cada fuente, además de las listas de códigos
candidatos por fuente.

**Justificación metodológica:** este archivo es el filtro maestro — define qué indicadores
entran al dataset unificado. Usamos `groq_dimension_principal` (clasificación vigente, ya con
las reclasificaciones manuales incorporadas), no `groq_dimension_principal_original`.

**Resultado esperado:** `df_metadata_indicadores` con 1.055 filas (975 WDI + 79 V-Dem + 1 MC) y
3 columnas: `codigo`, `fuente`, `dimension_principal`.


In [3]:
df_llm = pd.read_excel("LLM_dataset_final.xlsx")

df_metadata_indicadores = df_llm[["codigo", "fuente", "dimension_principal"]].rename(
    columns={"groq_dimension_principal": "dimension_principal"}
)

codigos_wdi = df_metadata_indicadores.loc[df_metadata_indicadores["fuente"] == "WDI", "codigo"].tolist()
codigos_vdem = df_metadata_indicadores.loc[df_metadata_indicadores["fuente"] == "V-Dem", "codigo"].tolist()
codigos_mc = df_metadata_indicadores.loc[df_metadata_indicadores["fuente"] == "MC", "codigo"].tolist()

print(f"Metadata de indicadores: {df_metadata_indicadores.shape}")
print(f"WDI candidatos: {len(codigos_wdi)}")
print(f"V-Dem candidatos: {len(codigos_vdem)}")
print(f"MC candidatos: {len(codigos_mc)}")

assert df_metadata_indicadores["codigo"].is_unique, "Hay codigos duplicados en la metadata"


Metadata de indicadores: (1055, 3)
WDI candidatos: 975
V-Dem candidatos: 79
MC candidatos: 1


**Qué comprobar:** debería dar 975 / 79 / 1. Si tu `LLM_dataset_final.xlsx` local difiere,
detené acá antes de seguir — el resto del cuaderno asume estos números.


## 3. Universo ONU (193 países)

**Objetivo:** cargar `df_regiones_miembros_onu.xlsx`, que es la tabla de referencia de los 193
miembros ONU con su `ISO-alpha3` y su `cow_code_countryVdem` — la misma que se usó en los
cuadernos 02, 06 y 09.

**Justificación metodológica:** usar siempre la misma tabla de referencia asegura que el
universo de países es idéntico en las tres fuentes, evitando inconsistencias entre cuadernos.

**Resultado esperado:** `df_onu` con 193 filas.


In [4]:
df_onu = pd.read_excel("df_regiones_miembros_onu.xlsx")

cols_onu_id = [
    "Country or Area", "ISO-alpha3", "M49_region", "Region Name",
    "M49_subregion", "Sub-region Name", "cow_code_countryVdem",
]
df_onu = df_onu[cols_onu_id].copy()

iso_onu = set(df_onu["ISO-alpha3"])

print(f"Paises ONU de referencia: {df_onu.shape[0]}")
assert df_onu.shape[0] == 193, "El universo ONU no tiene 193 filas — revisar df_regiones_miembros_onu.xlsx"


Paises ONU de referencia: 193


## 4. Procesamiento V-Dem

**Objetivo:** partir de `df_VDEM_base.xlsx` (ya extraído, ya filtrado a ONU, formato ancho con
79 columnas de indicadores), filtrar años 2000–2020, anexar `ISO-alpha3` y convertir a formato
largo.

**Justificación metodológica:** `df_VDEM_base.xlsx` ya pasó por el proceso de extracción desde
el CSV crudo de V-Dem (~397MB) en el cuaderno 06, con el universo ONU ya aplicado (172/193 con
match por `cow_code_countryVdem`). No hace falta volver a leer el CSV crudo — minimiza tiempo de
cómputo y reutiliza un artefacto ya validado. Solo falta traer `ISO-alpha3` (no está en el
archivo original) para unificar la clave de país con WDI y MC.

**Resultado esperado:** `df_vdem_largo` en formato largo, con columnas
`ISO-alpha3, Country or Area, M49_region, Region Name, M49_subregion, Sub-region Name, year,
codigo, valor`.


In [5]:
df_vdem_base = pd.read_excel("df_VDEM_base.xlsx", sheet_name="df_VDEM_base")

df_vdem_base = df_vdem_base[
    (df_vdem_base["year"] >= 2000) & (df_vdem_base["year"] <= 2020)
].copy()

print(f"Shape tras filtro de anios (2000-2020): {df_vdem_base.shape}")

# Auditoria: codigos V-Dem candidatos que NO estan como columna en df_VDEM_base
codigos_vdem_faltantes = [c for c in codigos_vdem if c not in df_vdem_base.columns]
codigos_vdem_presentes = [c for c in codigos_vdem if c in df_vdem_base.columns]

print(f"Codigos V-Dem candidatos presentes en df_VDEM_base: {len(codigos_vdem_presentes)} / {len(codigos_vdem)}")
if codigos_vdem_faltantes:
    print(f"ATENCION - codigos V-Dem candidatos NO encontrados (se documentan, no se inventan): {codigos_vdem_faltantes}")


Shape tras filtro de anios (2000-2020): (3601, 88)
Codigos V-Dem candidatos presentes en df_VDEM_base: 79 / 79


In [6]:
# Anexar ISO-alpha3 (no viene en df_VDEM_base) via cow_code_countryVdem
df_vdem_base = df_vdem_base.merge(
    df_onu[["cow_code_countryVdem", "ISO-alpha3"]],
    on="cow_code_countryVdem",
    how="left",
)

sin_iso3_vdem = df_vdem_base["ISO-alpha3"].isna().sum()
print(f"Filas sin match de ISO-alpha3 tras el merge: {sin_iso3_vdem}")
assert sin_iso3_vdem == 0, "Hay filas de V-Dem sin ISO-alpha3 asignado — revisar cow_code_countryVdem"


Filas sin match de ISO-alpha3 tras el merge: 0


In [7]:
id_vars_vdem = [
    "ISO-alpha3", "Country or Area", "M49_region", "Region Name",
    "M49_subregion", "Sub-region Name", "year",
]

df_vdem_largo = df_vdem_base.melt(
    id_vars=id_vars_vdem,
    value_vars=codigos_vdem_presentes,
    var_name="codigo",
    value_name="valor",
)

print(f"Shape df_vdem_largo: {df_vdem_largo.shape}")
print(f"Paises unicos: {df_vdem_largo['ISO-alpha3'].nunique()}")
print(f"Indicadores unicos: {df_vdem_largo['codigo'].nunique()}")
df_vdem_largo.head()


Shape df_vdem_largo: (284479, 9)
Paises unicos: 172
Indicadores unicos: 79


,ISO-alpha3,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,year,codigo,valor
0,USA,United States of America,19,Americas,21,Northern America,2000,v2x_suffr,1.0
1,USA,United States of America,19,Americas,21,Northern America,2001,v2x_suffr,1.0
2,USA,United States of America,19,Americas,21,Northern America,2002,v2x_suffr,1.0
3,USA,United States of America,19,Americas,21,Northern America,2003,v2x_suffr,1.0
4,USA,United States of America,19,Americas,21,Northern America,2004,v2x_suffr,1.0


**Qué comprobar:**
- `codigos_vdem_faltantes` debería salir vacío (los 79 códigos de `LLM_dataset_final` deberían
  coincidir exactamente con las columnas de `df_VDEM_base.xlsx`, ya que ambos vienen del mismo
  archivo `metadatavdem_79_indices_intermedios.xlsx`). Si no está vacío, hay una desincronización
  entre `LLM_dataset_final.xlsx` y la extracción V-Dem que hay que resolver antes de seguir.
- `sin_iso3_vdem` debe ser 0.
- Filas esperadas ≈ (países V-Dem con dato en 2000-2020) × 21 años × 79 indicadores.


## 5. Procesamiento Market Concentration

**Objetivo:** partir de `market_concentration_base.xlsx` (base cruda, 209 códigos `reporter_code`
sin recodificar), recodificar los códigos legacy/históricos usando
`estandarizar_iso3_codhistorico.xlsx` (mismo procedimiento que en el cuaderno 09), unir con el
universo ONU y filtrar años 2000–2020.

**Justificación metodológica:** sin recodificar, el match contra ONU es 178/193. Recodificando
los 6 códigos legacy (ROM, ZAR, SER, SUD, TMP, MNT) a su ISO3 vigente, sube a 183/193 (10 países
sin dato real de HHI, documentados en el cuaderno 09). Reutilizamos exactamente ese
procedimiento ya validado para no introducir inconsistencias entre cuadernos.

**Resultado esperado:** `df_mc_largo` con las mismas columnas que V-Dem, un único `codigo`
(`hh_market_concentration`).


In [8]:
df_mc = pd.read_excel("market_concentration_base.xlsx")

print(f"Shape df_mc (crudo): {df_mc.shape}")
print(f"Codigos reporter_code unicos (crudo): {df_mc['reporter_code'].nunique()}")


Shape df_mc (crudo): (5129, 3)
Codigos reporter_code unicos (crudo): 209


In [9]:
# Recodificacion de codigos legacy/historicos a ISO3 vigente (mismo procedimiento del cuaderno 09)
df_codigos_historicos = pd.read_excel("estandarizar_iso3_codhistorico.xlsx", sheet_name="codigos_historicos")

mapa_valido = df_codigos_historicos[
    df_codigos_historicos["Comtrade"].str.match(r"^[A-Z]{3}$", na=False)
]
codigos_legacy_a_iso3 = dict(zip(mapa_valido["WITS"], mapa_valido["Comtrade"]))

codigos_legacy_presentes = set(df_mc["reporter_code"]) & set(codigos_legacy_a_iso3)
n_filas_afectadas = df_mc["reporter_code"].isin(codigos_legacy_a_iso3).sum()

df_mc["reporter_code"] = df_mc["reporter_code"].replace(codigos_legacy_a_iso3)

print(f"Codigos legacy recodificados: {sorted(codigos_legacy_presentes)}")
print(f"Filas recodificadas: {n_filas_afectadas}")
print(f"Codigos unicos tras recodificacion: {df_mc['reporter_code'].nunique()}")

dups_mc = df_mc.duplicated(subset=["reporter_code", "year"]).sum()
assert dups_mc == 0, "Hay duplicados (reporter_code, year) tras la recodificacion — revisar"
print("Verificacion de duplicados OK.")


Codigos legacy recodificados: ['MNT', 'ROM', 'SER', 'SUD', 'TMP', 'ZAR']
Filas recodificadas: 106
Codigos unicos tras recodificacion: 208
Verificacion de duplicados OK.


In [10]:
# Union con universo ONU (inner: solo paises ONU con dato de MC) + filtro de anios
df_mc_onu = df_onu.merge(
    df_mc, left_on="ISO-alpha3", right_on="reporter_code", how="inner",
)

df_mc_onu = df_mc_onu[
    (df_mc_onu["year"] >= 2000) & (df_mc_onu["year"] <= 2020)
].copy()

print(f"Shape df_mc_onu (ONU + 2000-2020): {df_mc_onu.shape}")
print(f"Paises unicos: {df_mc_onu['ISO-alpha3'].nunique()} de 193")


Shape df_mc_onu (ONU + 2000-2020): (3373, 10)
Paises unicos: 182 de 193


In [11]:
# Verificacion: el codigo de LLM_dataset_final para MC debe coincidir con el nombre de columna real
assert len(codigos_mc) == 1, "Se esperaba un unico codigo MC en LLM_dataset_final"
codigo_mc = codigos_mc[0]
assert codigo_mc in df_mc_onu.columns, f"El codigo '{codigo_mc}' no coincide con ninguna columna de df_mc_onu"

df_mc_largo = df_mc_onu.rename(columns={codigo_mc: "valor"}).copy()
df_mc_largo["codigo"] = codigo_mc

cols_finales = [
    "ISO-alpha3", "Country or Area", "M49_region", "Region Name",
    "M49_subregion", "Sub-region Name", "year", "codigo", "valor",
]
df_mc_largo = df_mc_largo[cols_finales]

print(f"Shape df_mc_largo: {df_mc_largo.shape}")
df_mc_largo.head()


Shape df_mc_largo: (3373, 9)


,ISO-alpha3,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,year,codigo,valor
9,USA,United States of America,19,Americas,21,Northern America,2000,hh_market_concentration,0.076019
10,USA,United States of America,19,Americas,21,Northern America,2001,hh_market_concentration,0.072339
11,USA,United States of America,19,Americas,21,Northern America,2002,hh_market_concentration,0.072139
12,USA,United States of America,19,Americas,21,Northern America,2003,hh_market_concentration,0.071134
13,USA,United States of America,19,Americas,21,Northern America,2004,hh_market_concentration,0.067992


**Qué comprobar:**
- `codigos_legacy_presentes` debería mostrar los mismos 6 códigos que en el cuaderno 09.
- Países únicos tras el join ONU debería rondar 183 (puede variar levemente según cobertura
  2000-2020 específica vs. el rango completo usado en el cuaderno 09).
- El assert de `codigo_mc in df_mc_onu.columns` es el chequeo de que el nombre de indicador en
  `LLM_dataset_final.xlsx` (`hh_market_concentration`) coincide con la columna real de
  `market_concentration_base.xlsx`. Si falla, hay que revisar si cambiaste el nombre en algún
  archivo.


## 6. Procesamiento WDI

**Objetivo:** leer el CSV crudo de WDI (bulk download del Banco Mundial, mismo patrón del
cuaderno 02), filtrar por los 975 códigos candidatos y por el universo ONU, y convertir a
formato largo.

**Justificación metodológica:** a diferencia de V-Dem y MC, no existe un archivo intermedio con
los valores WDI ya extraídos (el cuaderno 02 solo exportó métricas de disponibilidad, no los
datos). Por eso este bloque vuelve a descargar/leer el ZIP bulk oficial, igual que en el
cuaderno 02 — es la única forma reproducible de obtener los valores.

**Resultado esperado:** `df_wdi_largo`, mismas columnas que las otras dos fuentes.

**Aviso:** este bloque puede tardar varios minutos (el CSV de WDI es grande) y requiere conexión
a internet la primera vez que se ejecuta (no en corridas posteriores, si `WDICSV.CSV` ya está en
`data/raw/`).


In [12]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

WDI_ZIP_URL = "https://databankfiles.worldbank.org/public/ddpext_download/WDI_CSV.zip"
zip_path = RAW_DIR / "WDI_csv.zip"

if zip_path.exists():
    print(f"Ya existe {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB). No se vuelve a descargar.")
else:
    resp = requests.get(WDI_ZIP_URL, timeout=120)
    resp.raise_for_status()
    zip_path.write_bytes(resp.content)
    print(f"Descargado {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")


Ya existe data\raw\WDI_csv.zip (283.2 MB). No se vuelve a descargar.


In [13]:
with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    data_file = [n for n in names if n.upper().endswith("WDICSV.CSV")][0]
    z.extract(data_file, RAW_DIR)

data_path = RAW_DIR / data_file
print(f"Archivo de datos WDI: {data_path}")


Archivo de datos WDI: data\raw\WDICSV.csv


In [14]:
wdi_raw = pd.read_csv(data_path, low_memory=False)

wdi = wdi_raw[
    wdi_raw["Country Code"].isin(iso_onu)
    & wdi_raw["Indicator Code"].isin(codigos_wdi)
].copy()

print(f"Filas tras filtro (ONU + indicadores candidatos): {len(wdi)}")
print(f"Indicadores presentes: {wdi['Indicator Code'].nunique()} / {len(codigos_wdi)}")
print(f"Paises presentes: {wdi['Country Code'].nunique()} / 193")

# Auditoria: codigos WDI candidatos que NO aparecen en el CSV crudo
codigos_wdi_faltantes = sorted(set(codigos_wdi) - set(wdi["Indicator Code"].unique()))
if codigos_wdi_faltantes:
    print(f"ATENCION - {len(codigos_wdi_faltantes)} codigos WDI candidatos no encontrados en WDICSV.CSV:")
    print(codigos_wdi_faltantes)


Filas tras filtro (ONU + indicadores candidatos): 188175
Indicadores presentes: 975 / 975
Paises presentes: 193 / 193


In [15]:
year_cols = [c for c in wdi.columns if c.strip().isdigit()]

wdi_largo = wdi.melt(
    id_vars=["Country Code", "Indicator Code"],
    value_vars=year_cols,
    var_name="year",
    value_name="valor",
)
wdi_largo["year"] = wdi_largo["year"].astype(int)

wdi_largo = wdi_largo[(wdi_largo["year"] >= 2000) & (wdi_largo["year"] <= 2020)].copy()

print(f"Shape tras melt + filtro de anios (2000-2020): {wdi_largo.shape}")


Shape tras melt + filtro de anios (2000-2020): (3951675, 4)


In [16]:
# Anexar identificadores de pais/region (misma tabla ONU usada en las otras dos fuentes)
df_wdi_largo = wdi_largo.merge(
    df_onu, left_on="Country Code", right_on="ISO-alpha3", how="left",
)

df_wdi_largo = df_wdi_largo.rename(columns={"Indicator Code": "codigo"})

cols_finales = [
    "ISO-alpha3", "Country or Area", "M49_region", "Region Name",
    "M49_subregion", "Sub-region Name", "year", "codigo", "valor",
]
df_wdi_largo = df_wdi_largo[cols_finales]

print(f"Shape df_wdi_largo: {df_wdi_largo.shape}")
df_wdi_largo.head()


Shape df_wdi_largo: (3951675, 9)


,ISO-alpha3,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,year,codigo,valor
0,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,EG.ELC.ACCS.ZS,4.4
1,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,EG.ELC.ACCS.RU.ZS,NaN
2,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,EG.ELC.ACCS.UR.ZS,73.4
3,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,FX.OWN.TOTL.ZS,NaN
4,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,FX.OWN.TOTL.FE.ZS,NaN


**Qué comprobar:**
- `codigos_wdi_faltantes`: es esperable que no esté vacío (algunos códigos WDI se discontinúan o
  cambian de nombre entre versiones del bulk download). Documentalo, no lo ignores — si la lista
  es larga (más de un puñado), avisame antes de seguir para revisar si cambió la versión del
  dataset.
- `df_wdi_largo.shape[0]` debería aproximarse a 193 × 21 × (indicadores realmente presentes) —
  va a ser, por lejos, la fuente más pesada del dataset unificado.
- Verificá que no haya `NaN` en `ISO-alpha3` tras el merge final (si los hay, algún código de
  país de WDI no matcheó con la tabla ONU).


In [17]:
sin_iso3_wdi = df_wdi_largo["ISO-alpha3"].isna().sum()
print(f"Filas sin ISO-alpha3 tras el merge: {sin_iso3_wdi}")
assert sin_iso3_wdi == 0, "Hay filas de WDI sin ISO-alpha3 — revisar el join"


Filas sin ISO-alpha3 tras el merge: 0


## 7. Anexar metadata (`fuente`, `dimension_principal`) y unificar

**Objetivo:** anexar a cada base larga las columnas `fuente` y `dimension_principal` (desde
`df_metadata_indicadores`) y concatenar las tres fuentes en un único dataset.

**Justificación metodológica:** el cruce se hace por `codigo`, que es único a nivel global (ya
verificado en el bloque 2 — no hay colisión de códigos entre WDI, V-Dem y MC). Concatenar
después del merge asegura que cada fila del dataset final ya trae toda la metadata necesaria
para pivotar por dimensión en las etapas de PCA.

**Resultado esperado:** `df_unificado`, con columnas
`ISO-alpha3, Country or Area, M49_region, Region Name, M49_subregion, Sub-region Name, year,
codigo, fuente, dimension_principal, valor`.


In [18]:
def anexar_metadata(df_largo, nombre_fuente):
    df_out = df_largo.merge(df_metadata_indicadores, on="codigo", how="left")
    sin_metadata = df_out["fuente"].isna().sum()
    assert sin_metadata == 0, f"{nombre_fuente}: hay {sin_metadata} filas sin metadata tras el merge"
    return df_out

df_vdem_final = anexar_metadata(df_vdem_largo, "V-Dem")
df_mc_final = anexar_metadata(df_mc_largo, "MC")
df_wdi_final = anexar_metadata(df_wdi_largo, "WDI")

print("Metadata anexada correctamente en las tres fuentes.")


Metadata anexada correctamente en las tres fuentes.


In [19]:
df_unificado = pd.concat([df_wdi_final, df_vdem_final, df_mc_final], ignore_index=True)

print(f"Shape df_unificado: {df_unificado.shape}")
print(f"\nFilas por fuente:")
print(df_unificado["fuente"].value_counts())
print(f"\nFilas por dimension_principal:")
print(df_unificado["dimension_principal"].value_counts())
print(f"\nPaises unicos: {df_unificado['ISO-alpha3'].nunique()} de 193")
print(f"Rango de anios: {df_unificado['year'].min()} - {df_unificado['year'].max()}")
print(f"Indicadores unicos: {df_unificado['codigo'].nunique()}")


Shape df_unificado: (4239527, 11)

Filas por fuente:
fuente
WDI      3951675
V-Dem     284479
MC          3373
Name: count, dtype: int64

Filas por dimension_principal:
dimension_principal
SOCIODEMOGRAFICA    1977864
ESTRUCTURAL          995002
MONETARIA            798441
INSTITUCIONAL        468220
Name: count, dtype: int64

Paises unicos: 193 de 193
Rango de anios: 2000 - 2020
Indicadores unicos: 1055


In [22]:
df_unificado.columns

Index(['ISO-alpha3', 'Country or Area', 'M49_region', 'Region Name',
       'M49_subregion', 'Sub-region Name', 'year', 'codigo', 'valor', 'fuente',
       'dimension_principal'],
      dtype='str')

**Qué comprobar:**
- El total de indicadores únicos en `df_unificado` debería ser ≤ 1.055 (puede ser menor si
  hubo códigos faltantes documentados en los bloques 4 o 6).
- `Paises unicos` debería ser 193 si al menos un indicador de alguna fuente cubre a cada país —
  esto no implica que todos los países tengan dato en las tres fuentes.


## 8. Auditoría final de indicadores no incorporados

**Objetivo:** documentar explícitamente qué códigos de `LLM_dataset_final.xlsx` no llegaron al
dataset final, para no perderlos silenciosamente (principio: nunca descartar sin dejar rastro).

**Justificación metodológica:** las auditorías parciales de los bloques 4 y 6 ya identificaron
faltantes por fuente; este bloque las consolida en una única tabla de referencia.

**Resultado esperado:** `df_indicadores_no_incorporados`, con los códigos ausentes y su fuente.


In [20]:
codigos_incorporados = set(df_unificado["codigo"].unique())
codigos_candidatos_totales = set(df_metadata_indicadores["codigo"])

codigos_no_incorporados = sorted(codigos_candidatos_totales - codigos_incorporados)

df_indicadores_no_incorporados = df_metadata_indicadores[
    df_metadata_indicadores["codigo"].isin(codigos_no_incorporados)
].reset_index(drop=True)

print(f"Indicadores candidatos no incorporados al dataset final: {len(df_indicadores_no_incorporados)}")
df_indicadores_no_incorporados


Indicadores candidatos no incorporados al dataset final: 0


,codigo,fuente,dimension_principal


## 9. Exportación

**Objetivo:** guardar el dataset unificado en CSV (por el volumen de filas, no es viable en
`.xlsx`) y, opcionalmente, un resumen de auditoría en Excel.

**Justificación metodológica:** el CSV final es el insumo directo para los próximos cuadernos
(selección de variables, PCA por dimensión, clustering). Las celdas de exportación están
comentadas por defecto, siguiendo la convención ya usada en cuadernos anteriores.

**Resultado esperado:** `df_unificado.csv` en `data/processed/`, y opcionalmente
`13_auditoria_unificacion.xlsx` con la tabla de indicadores no incorporados.


In [ ]:
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Exportacion del dataset unificado (formato largo) a CSV ---
out_path_csv = OUT_DIR / "df_unificado.csv"
df_unificado.to_csv(out_path_csv, index=False)
print(f"Exportado: {out_path_csv} ({df_unificado.shape[0]} filas)")

# # --- Exportacion opcional de la auditoria de indicadores no incorporados ---
# out_path_auditoria = OUT_DIR / "13_auditoria_unificacion.xlsx"
# with pd.ExcelWriter(out_path_auditoria, engine="openpyxl") as writer:
#     df_indicadores_no_incorporados.to_excel(writer, sheet_name="no_incorporados", index=False)
# print(f"Exportado: {out_path_auditoria}")


Exportado: data\processed\df_unificado.csv (4239527 filas)
Exportado: data\processed\13_auditoria_unificacion.xlsx
